# Elasticsearch Basics in Python

This notebook demonstrates how to connect to Elasticsearch from Python, check the connection, create an index, insert documents, search, and delete data.

**Make sure your Elasticsearch and Jupyter containers are running!**

In [ ]:
from elasticsearch import Elasticsearch

# Connect to Elasticsearch running in Docker
es = Elasticsearch(
    'http://elasticsearch:9200',
    basic_auth=('elastic', '111')  # Replace with your actual password
)

# Test connection
if es.ping():
    print('Connected to Elasticsearch!')
else:
    print('Could not connect to Elasticsearch.')

## Create an Index
Let's create a simple index called `test-index`.

In [ ]:
index_name = 'test-index'

# Create the index (ignore if it exists)
if not es.indices.exists(index=index_name):
    es.indices.create(index=index_name)
    print(f'Index `{index_name}` created.')
else:
    print(f'Index `{index_name}` already exists.')

## Index (Insert) a Document
Let's add a simple document to our index.

In [ ]:
es.index(index="movies", id=1, document={"title": "Inception",    "director": "Christopher Nolan", "description": "A thief using dream-sharing technology."})
es.index(index="movies", id=2, document={"title": "Interstellar", "director": "Christopher Nolan", "description": "Explorers travel through space using advanced technology."})
es.indices.refresh(index="movies")
print("Documents prêts")

## 🔎 Recherche de Documents

Elasticsearch utilise un **index inversé** : chaque mot est associé à la liste des documents où il apparaît.

- **Mot partagé** → le mot existe dans plusieurs documents → **plusieurs résultats**
- **Mot unique** → le mot n'existe que dans un document → **un seul résultat**


In [ ]:
# ─── MOT PARTAGÉ ────────────────────────────────────────────────────────────
# "Nolan" apparaît dans le champ "director" des 2 documents
# → Elasticsearch retourne les 2 documents complets

res = es.search(index="movies", query={"match": {"director": "Nolan"}})

print(f" Recherche 'Nolan' → {res['hits']['total']['value']} résultat(s)\n")
for h in res["hits"]["hits"]:
    src = h["_source"]
    print(f"     Titre      : {src['title']}")
    print(f"     Réalisateur: {src['director']}")
    print(f"     Description: {src['description']}")
    print()

In [ ]:
# ─── MOT UNIQUE ─────────────────────────────────────────────────────────────
# "dream" apparaît UNIQUEMENT dans la description d'Inception
# → Elasticsearch retourne 1 seul document complet

res = es.search(index="movies", query={"match": {"description": "dream"}})

print(f"  Recherche 'dream' → {res['hits']['total']['value']} résultat(s)\n")
for h in res["hits"]["hits"]:
    src = h["_source"]
    print(f"     Titre      : {src['title']}")
    print(f"     Réalisateur: {src['director']}")
    print(f"     Description: {src['description']}")
    print()

In [ ]:

# --- SCORING DIFFERENT ------
# "technology" apparaît dans les descriptions des 2 films
# Le score d'Inception est plus élévé, sûrement car ....

res = es.search(index="movies", query={"match": {"description": "technology"}})

for hit in res["hits"]["hits"]:
    print(f"Score: {hit['_score']}, Title: {hit['_source'].get('title')}")

In [ ]:
# --- BOOST SCORING ------
# On peut influencer le score en boostant certains critères

query_boost = {
    "bool": {
        "should": [
            {"match": {"title": {"query": "Inception", "boost": 3}}},
            {"match": {"description": {"query": "technology", "boost": 1}}}
        ]
    }
}

res = es.search(index="movies", query=query_boost)

for hit in res["hits"]["hits"]:
    print(f"Score: {hit['_score']}, Title: {hit['_source'].get('title')}")

In [ ]:
# --- FUZZY MATCH ------
# Permet de trouver des correspondances proches (typos, variantes)

query_clean = {
    "match": {
        "title": {
            "query": "Interstellar",
        }
    }
}
query_typo = {
    "match": {
        "title": {
            "query": "Interstllar",  # typo
            "fuzziness": "AUTO"
        }
    }
}

res_clean = es.search(index="movies", query=query_clean)
print("Résultats pour 'Interstellar' (clean):")
for hit in res_clean["hits"]["hits"]:
    print(f"Score: {hit['_score']}, Title: {hit['_source'].get('title')}")


res_typo = es.search(index="movies", query=query_typo)
print("\nRésultats pour 'Interstllar' (typo):")
for hit in res_typo["hits"]["hits"]:
    print(f"Score: {hit['_score']}, Title: {hit['_source'].get('title')}")

In [ ]:
# --- PERFORMANCE ---

INDEX_NAME = "ncedc-earthquakes-earthquake"

res = es.search(index=INDEX_NAME, query={
        "match": {
            "type": "earthquake"
        }
    })

print("Query time:", res["took"], "ms")
print("Total hits:", res["hits"]["total"]["value"])

## Delete the Index
Let's clean up by deleting the index.

In [ ]:
es.indices.delete(index="movies")
print(f'Index `{"movies"}` deleted.')

es.indices.delete(index="test-index")